# Detekcija spam SMS poruka

Ova sveska je finalni demo projekta. Projekat koristi UCI SMS Spam Collection skup podataka, klasicni NLP model `TF-IDF + Logistic Regression` i dodatni `LSTM` rekurentni model.

## 1. Ucitavanje pripremljenih podataka

Podaci su vec podeljeni na trening, validacioni i test skup. Test skup se ne koristi za izbor modela.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

train_df = pd.read_csv(ROOT / "data" / "processed" / "splits" / "train.csv")
validation_df = pd.read_csv(ROOT / "data" / "processed" / "splits" / "validation.csv")
test_df = pd.read_csv(ROOT / "data" / "processed" / "splits" / "test.csv")

pd.DataFrame({
    "skup": ["train", "validation", "test"],
    "broj_poruka": [len(train_df), len(validation_df), len(test_df)],
    "ham": [sum(train_df["label"] == "ham"), sum(validation_df["label"] == "ham"), sum(test_df["label"] == "ham")],
    "spam": [sum(train_df["label"] == "spam"), sum(validation_df["label"] == "spam"), sum(test_df["label"] == "spam")],
})

## 2. Osnovna analiza

Osnovnu analizu racunamo direktno iz pripremljenih podataka.

In [ ]:
clean_df = pd.read_csv(ROOT / "data" / "processed" / "sms_spam_clean.csv")
modeling_df = pd.read_csv(ROOT / "data" / "processed" / "sms_spam_modeling.csv")

class_counts = clean_df["label"].value_counts()
class_percentages = (clean_df["label"].value_counts(normalize=True) * 100).round(2)
duplicate_count = clean_df.duplicated(["label", "message"]).sum()
length_stats = clean_df.groupby("label")[["char_count", "word_count", "digit_count"]].mean().round(2)

display(pd.DataFrame({"broj_poruka": class_counts, "procenat": class_percentages}))
print(f"Broj dupliranih poruka: {duplicate_count}")
print(f"Broj poruka posle uklanjanja duplikata: {len(modeling_df)}")
display(length_stats)

## 3. Rezultati TF-IDF + Logistic Regression modela

Ovaj model je glavni model projekta. Hiperparametri su izabrani pomocu validacionog skupa.

In [ ]:
print((ROOT / "reports" / "03_tfidf_logreg_results.txt").read_text(encoding="utf-8"))

## 4. Rezultati LSTM modela

LSTM je dodat kao rekurentni model. Evaluacija je uradjena na istom validacionom skupu.

In [ ]:
print((ROOT / "reports" / "05_lstm_results.txt").read_text(encoding="utf-8"))

## 5. Poredjenje modela

Poredjenje modela moze da se uradi rucno na osnovu metrika iz prethodna dva izvestaja. U ovom projektu glavni model je `TF-IDF + Logistic Regression`, jer je jednostavniji, brzi i ima veci F1 za spam klasu na validacionom skupu.

## 6. Demo predikcije

U ovom delu se ucitava sacuvani TF-IDF model i proverava nekoliko novih poruka.

In [ ]:
import joblib

model = joblib.load(ROOT / "models" / "tfidf_logreg_pipeline.joblib")

messages = [
    "Hey, are we still meeting at 6 tonight?",
    "Congratulations! You won a free prize. Call now to claim cash.",
    "Can you send me the notes from class?",
    "URGENT! Your account has won 1000 pounds, reply WIN now.",
]

predictions = model.predict(messages)
pd.DataFrame({"message": messages, "prediction": predictions})